# IR+PE+NOM 最大ドメインの探索ベース地図化（phi^IP 非依存）

**論文**: Manjunath & Westkamp (2025) *Marginal Mechanisms For Balanced Exchange* (arXiv:2502.06499)

**問い**: 小スケール（n=3, 保有財数 k≤3）で **IR+PE+NOM を満たす marginal mechanism が存在する最大ドメイン D_NOM** はどこか。
論文 Theorem 1（IR+PE の最大ドメイン=trichotomous）は **各自≥4財** が前提で、この regime はどの定理も適用外。

**鍵**: D_NOM は「機構が *存在* するか」という存在問題。phi^IP は witness の1つにすぎず**不要**。機構を直接**探索**する。
- witness を1つ見つける → feasibility を**証明**（健全）。
- 見つからない → infeasibility の証明にはならない（完全探索/理論が必要）。
- IR+PE が空のプロファイルが1つでもあれば → **infeasibility を証明**（機構フリー・安価）。

3つの入れ子: `D_SP ⊆ D_NOM ⊆ D_IRPE`。

In [1]:
# Colab 設定。リポジトリ名/ブランチは適宜修正。
!git clone -q https://github.com/shiro-seminar/NOM-matching 2>/dev/null || echo "(cloned or set path)"
%cd NOM-matching
import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

/content/NOM-matching
device: cuda


## 実験A — IR+PE 最大ドメイン D_IRPE は config 依存（機構フリー・決定的）

各候補ドメインで「全プロファイルに unambiguous IR+PE マッチングが存在するか」を判定（空が出れば infeasible）。
保有財数 k を増やすと richer ドメインが脱落し、D_IRPE が trichotomous へ収束する様子を見る。

In [2]:
from domain_expansion_experiments.config import Config
from domain_expansion_experiments.domains import DOMAINS, domain_lattice
from domain_frontier.feasibility import domain_feasible

# repo の標準 config（n=3, m=4, 各自1-2財）: 全ドメイン feasible（k<4 では obstruction が組めない）
print("=== config: n=3, m=4 (各自1-2財) ===")
for dom in domain_lattice():
    cfg = Config(domain=dom.name)
    res = domain_feasible(cfg, dom, verbose=False)
    print(f"  {dom.name:26s}: IR+PE feasible = {res['feasible']}  (empties={res['n_empty']}, {res['mode']})")

=== config: n=3, m=4 (各自1-2財) ===
  strongly_tri              : IR+PE feasible = True  (empties=0, enumerate)
  trichotomous              : IR+PE feasible = True  (empties=0, enumerate)
  trichotomous_extended_e3  : IR+PE feasible = True  (empties=0, sample)
  four_chotomous_e4         : IR+PE feasible = True  (empties=0, sample)


In [3]:
# 保有財数を増やす: m=6 (各自2財), m=9 (各自3財) で four_chotomous が脱落するかを確認
from domain_frontier.feasibility import domain_feasible
for m in (6, 9):
    print(f"\n=== config: n=3, m={m} (各自{m//3}財), four_chotomous_e4 ===")
    cfg = Config(domain="four_chotomous_e4", num_items=m)
    res = domain_feasible(cfg, DOMAINS["four_chotomous_e4"], n_samples=20000, verbose=False)
    print(f"  IR+PE feasible = {res['feasible']}  (empties={res['n_empty']}, mode={res['mode']})")
    if res["example_empty"]:
        print(f"  empty profile witness: {res['example_empty']}")


=== config: n=3, m=6 (各自2財), four_chotomous_e4 ===
  IR+PE feasible = False  (empties=12886, mode=sample)
  empty profile witness: {'k_e': 14, 'profile': [[1, 1, 2, 0, 2, 3], [3, 1, 2, 0, 1, 3], [1, 3, 2, 3, 1, 3]]}

=== config: n=3, m=9 (各自3財), four_chotomous_e4 ===


KeyboardInterrupt: 

**読み**: m=4 では four_chotomous まで feasible（D_IRPE ⊋ trichotomous）。m=6,9 で four_chotomous は空プロファイルを生じ
脱落（既に各々 ~4/3000, ~6/3000 を確認済）。→ **D_IRPE は財数増で trichotomous へ収束**。境界の novel な地図。

## 実験B — unambiguous-SP 地図（D_SP の分離）

参照機構 `priority_mechanism` の unambiguous-SP 違反率。strongly_tri=0（Thm3 一致）、richer で増加。
trichotomous は後述の通り NOM witness を持つ（NOM=0）が SP 違反>0 → **D_SP ⊊ D_NOM** を実証。

In [4]:
from domain_frontier.sp_test import unamb_sp_violation_rate
from domain_expansion_experiments.benchmarks import priority_mechanism
for dom in domain_lattice():
    cfg = Config(domain=dom.name)
    r = unamb_sp_violation_rate(cfg, dom, priority_mechanism, n_profiles=400, seed=0)
    print(f"  {dom.name:26s}: SP-viol = {r['sp_viol']}/{r['n']}")
# 期待: strongly_tri 0, trichotomous ~11, four_chotomous ~88 (上昇)

  strongly_tri              : SP-viol = 0/400
  trichotomous              : SP-viol = 11/400
  trichotomous_extended_e3  : SP-viol = 36/400


KeyboardInterrupt: 

## 実験C — NOM witness 探索（中核・phi^IP 非依存）

`AllocationNet` を **hard な unambiguous IR+PE マスク**で制約しつつ NOM 代理損失で学習 → **full-enum FOSD オラクルで検証**。
検証 NOM=0 なら **witness 確定 = そのドメインは D_NOM に属する**。これは学習機構を FOSD-NOM で初めて検証する実験
（`PROJECT_OVERVIEW.md` 不明瞭点E の解消）。GPU 推奨。

In [ ]:
from domain_frontier.search_nom import train_witness, verify_nom_fullenum
dev = "cuda" if torch.cuda.is_available() else "cpu"

# trichotomous: witness が取れるはず（NOM=0）。SP 違反>0 と合わせ D_SP ⊊ D_NOM。
cfg = Config(domain="trichotomous", steps=3000, S=8, M=8, batch_size=256, device=dev)
net = train_witness(cfg, verbose=True); net.eval()
res = verify_nom_fullenum(cfg, DOMAINS["trichotomous"], net, device=dev, verbose=False)
print(f"trichotomous: witness={'FOUND' if res['witness'] else 'NO'} "
      f"(viol cells {res['nom_viol_cells']}/{res['total_cells']})")

In [ ]:
# four_chotomous_e4 (m=4): richer than trichotomous。witness が取れれば D_NOM が tri を超える証拠（phi^IP 抜き）。
# 注意: full-enum 検証は 16M evals/cell と重い。GPU 推奨。長い場合は endow_list を絞る。
cfg = Config(domain="four_chotomous_e4", steps=4000, S=8, M=8, batch_size=256, device=dev)
net4 = train_witness(cfg, verbose=True); net4.eval()
res4 = verify_nom_fullenum(cfg, DOMAINS["four_chotomous_e4"], net4, device=dev,
                           chunk=131072, verbose=True)
print(f"four_chotomous_e4: witness={'FOUND' if res4['witness'] else 'NO'} "
      f"(viol cells {res4['nom_viol_cells']}/{res4['total_cells']})")

## 実験D（任意）— 一括地図ドライバ

`map_frontier` で実験A/B（＋`--nom` でC）を一括実行し JSON 出力。

In [ ]:
!python -m domain_frontier.map_frontier --device cuda --sp_n 400
# NOM 列も含める場合（重い）:
# !python -m domain_frontier.map_frontier --nom --steps 3000 --S 8 --M 8 --batch 256 --device cuda

---
### まとめ
- **実験A**: D_IRPE は m=4 で four_chotomous まで含む（⊋ trichotomous）が、財数増で trichotomous に収束。
- **実験B**: SP 違反は richer で増加。trichotomous は NOM witness ありで SP 違反>0 → **D_SP ⊊ D_NOM**。
- **実験C**: phi^IP を実装せず、学習＋FOSD 検証で IR+PE+NOM 機構の witness を獲得。four_chotomous(m=4) で取れれば
  **小スケールで D_NOM が trichotomous を超える**ことを構成的に示す。

## 実験E — D_IRPE は「保有の形(shape)」に依存する（GPU・核心）

対称性より IR+PE feasibility は endowment の **サイズ shape のみ** で決まる（同 shape は object/agent 置換で同型なので代表1つで厳密）。

**発見**: シングルトン保有者が2人いる形 (k,1,1) では最大ドメインが **full（全弱順序）** に達し、それ以外（(3,2,1),(2,2,2),...）では **trichotomous** に落ちる。

下のセルは m=4,6,8,10 を一括で地図化し、(k,1,1) で richer が通り続けるか（characterization の確認）を GPU で検証する。

> 'feasible' は「空が見つからない」（サンプリング＝証明ではない）／'INFEASIBLE' は空発見＝証明。

In [5]:
from domain_frontier.size_sweep import sweep_by_m
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
# richness_lattice が R-chotomous を自動生成。m財なら最大 m クラスまでが順序的に有効。
sweep_by_m(ms=[4, 6, 8, 10], max_R=8, n_samples=20000, device=dev)


############ total objects m=4 (n=3) ############
  strongly_tri:
    shape (2, 1, 1)     : feasible
  trichotomous:
    shape (2, 1, 1)     : feasible
  trichotomous_extended_e3:
    shape (2, 1, 1)     : feasible
  four_chotomous_e4:
    shape (2, 1, 1)     : feasible
  5_chotomous:
    shape (2, 1, 1)     : feasible
  6_chotomous:
    shape (2, 1, 1)     : feasible
  7_chotomous:
    shape (2, 1, 1)     : feasible
  8_chotomous:
    shape (2, 1, 1)     : feasible
  --- MAX D_IRPE(m=4) per shape ---
    shape (2, 1, 1)     : MAX = 8_chotomous

############ total objects m=6 (n=3) ############
  strongly_tri:
    shape (4, 1, 1)     : feasible
    shape (3, 2, 1)     : feasible
    shape (2, 2, 2)     : feasible
  trichotomous:
    shape (4, 1, 1)     : feasible
    shape (3, 2, 1)     : feasible
    shape (2, 2, 2)     : feasible
  trichotomous_extended_e3:
    shape (4, 1, 1)     : feasible
    shape (3, 2, 1)     : INFEASIBLE (empties=33)
    shape (2, 2, 2)     : INFEASIBLE (empt

OutOfMemoryError: CUDA out of memory. Tried to allocate 21.05 GiB. GPU 0 has a total capacity of 39.49 GiB of which 17.94 GiB is free. Including non-PyTorch memory, this process has 21.54 GiB memory in use. Of the allocated memory 21.05 GiB is allocated by PyTorch, and 13.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)